In [1]:
# ==========================================
# PROJETO: TRATAMENTO DE DADOS DO YOUTUBE
# ==========================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, year, to_date, count, when, isnan
from pyspark.sql.types import LongType, IntegerType

# Inicializando a SparkSession
spark = SparkSession.builder.appName("TratamentoDeDadosYouTube").getOrCreate()

c:\Users\Paulo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
# ------------------------------------------
# PASSO 1: Ler o arquivo 'videos-stats.csv'
# ------------------------------------------
df_video = spark.read.csv('videos-stats.csv', header=True, inferSchema=True)

In [3]:
# ------------------------------------------
# PASSO 2: Alterar valores nulos de Likes, Comments e Views para 0
# ------------------------------------------
df_video = df_video.na.fill({'Likes': 0, 'Comments': 0, 'Views': 0})

In [4]:
# ------------------------------------------
# PASSO 3: Ler o arquivo 'comments.csv'
# ------------------------------------------
df_comentario = spark.read.csv(
    'comments.csv',
    header=True,
    inferSchema=True
)

In [5]:
# ------------------------------------------
# PASSO 4: Calcular a quantidade de registros
# ------------------------------------------
print("Quantidade de registros em df_video:", df_video.count())
print("Quantidade de registros em df_comentario:", df_comentario.count())

Quantidade de registros em df_video: 1881
Quantidade de registros em df_comentario: 30036


In [6]:
# ------------------------------------------
# PASSO 5: Remover registros com 'Video ID' nulo e recalcular
# ------------------------------------------
df_video = df_video.filter(col("Video ID").isNotNull())
df_comentario = df_comentario.filter(col("Video ID").isNotNull())

print("Registros em df_video após remover nulos:", df_video.count())
print("Registros em df_comentario após remover nulos:", df_comentario.count())

Registros em df_video após remover nulos: 1881
Registros em df_comentario após remover nulos: 22555


In [7]:
# ------------------------------------------
# PASSO 6: Remover registros duplicados de 'Video ID' apenas no df_video
# ------------------------------------------
df_video = df_video.dropDuplicates(["Video ID"])

In [8]:
# ------------------------------------------
# PASSO 7: Converter colunas Likes, Comments e Views para LongType (evita estouro de capacidade/overflow)
# ------------------------------------------
df_video = df_video \
    .withColumn("Likes", col("Likes").cast(LongType())) \
    .withColumn("Comments", col("Comments").cast(LongType())) \
    .withColumn("Views", col("Views").cast(LongType()))

In [9]:
# ------------------------------------------
# PASSO 8: Converter Likes e Sentiment para int em df_comentario e renomear Likes para 'Likes Comment'
# ------------------------------------------
from pyspark.sql.functions import expr
from pyspark.sql.types import IntegerType

df_comentario = df_comentario \
    .withColumn(
        "Likes",
        expr("try_cast(`Likes` AS DOUBLE)").cast(IntegerType())
    ) \
    .withColumn(
        "Sentiment",
        expr("try_cast(`Sentiment` AS DOUBLE)").cast(IntegerType())
    ) \
    .withColumnRenamed("Likes", "Likes Comment")

In [10]:
# ------------------------------------------
# PASSO 9: Criar o campo 'Interaction' no df_video (soma de Likes, Comments e Views)
# ------------------------------------------
df_video = df_video.withColumn("Interaction", col("Likes") + col("Comments") + col("Views"))

In [11]:
# ------------------------------------------
# PASSO 10: Converter o campo 'Published At' para 'date'
# ------------------------------------------
df_video = df_video.withColumn("Published At", to_date(col("Published At")))

In [12]:
# ------------------------------------------
# PASSO 11: Criar o campo 'Year' extraindo o ano de 'Published At'
# ------------------------------------------
df_video = df_video.withColumn("Year", year(col("Published At")))

In [13]:
# ==========================================
# PASSO 12: Mesclar df_comentario no df_video
# ==========================================

df_join_video_comments = df_video.join(
    df_comentario,
    ["Video ID"],
    "inner"
)

print("Total de registros:", df_join_video_comments.count())

primeiros_registros = df_join_video_comments.limit(5).collect()

for registro in primeiros_registros:
    print(registro)

Total de registros: 18409
Row(Video ID='wAZZ-UWGVHI', _c0=0, Title='Apple Pay Is Killing the Physical Wallet After Only Eight Years | Tech News Briefing Podcast | WSJ', Published At=datetime.date(2022, 8, 23), Keyword='tech', Likes=3407, Comments=672, Views=135612, Interaction=139691, Year=2022, _c0='0', Comment="Let's not forget that Apple Pay in 2014 required a brand new iPhone in order to use it.  A significant portion of Apple's user base wasn't able to use it even if they wanted to.  As each successive iPhone incorporated the technology and older iPhones were replaced the number of people who could use the technology increased.", Likes Comment=95, Sentiment=1)
Row(Video ID='wAZZ-UWGVHI', _c0=0, Title='Apple Pay Is Killing the Physical Wallet After Only Eight Years | Tech News Briefing Podcast | WSJ', Published At=datetime.date(2022, 8, 23), Keyword='tech', Likes=3407, Comments=672, Views=135612, Interaction=139691, Year=2022, _c0='1', Comment='Here in NZ 50% of retailers don’t eve

In [14]:
# ------------------------------------------
# PASSO 13: Ler o arquivo 'USvideos.csv' no dataframe 'df_us_videos'
# ------------------------------------------
df_us_videos = spark.read.csv('USvideos.csv', header=True, inferSchema=True)

In [15]:
# ------------------------------------------
# PASSO 14: Mesclar df_us_videos no df_video usando 'Title' (criando df_join_video_usvideos)
# (Renomeamos colunas conflitantes para evitar ambiguidade)
# ------------------------------------------
df_us_videos_clean = df_us_videos.withColumnRenamed("Views", "US_Views") \
                                   .withColumnRenamed("Likes", "US_Likes") \
                                   .withColumnRenamed("Comments", "US_Comments")

df_join_video_usvideos = df_video.join(df_us_videos_clean, on="Title", how="inner")
df_join_video_usvideos.show(5)

+--------------------+---+-----------+------------+-------+------+--------+---------+-----------+----+-----------+-------------+-------------+-----------+--------------------+--------------------+--------+--------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|               Title|_c0|   Video ID|Published At|Keyword| Likes|Comments|    Views|Interaction|Year|   video_id|trending_date|channel_title|category_id|        publish_time|                tags|US_Views|US_Likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+--------------------+---+-----------+------------+-------+------+--------+---------+-----------+----+-----------+-------------+-------------+-----------+--------------------+--------------------+--------+--------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------

In [17]:
# ------------------------------------------
# PASSO 15: Verificar quantidade de nulos
# em todos os campos do df_video
# ------------------------------------------

from pyspark.sql.functions import count, when, col, isnan
from pyspark.sql.types import FloatType, DoubleType

contagem_nulos = []

for campo in df_video.schema.fields:

    nome_coluna = campo.name

    if isinstance(campo.dataType, (FloatType, DoubleType)):

        condicao = col(nome_coluna).isNull() | isnan(col(nome_coluna))

    else:

        condicao = col(nome_coluna).isNull()

    contagem_nulos.append(
        count(when(condicao, 1)).alias(nome_coluna)
    )

resultado_nulos = df_video.select(contagem_nulos).collect()

for resultado in resultado_nulos:
    for coluna, quantidade in resultado.asDict().items():
        print(f"{coluna}: {quantidade} valores nulos")

_c0: 0 valores nulos
Title: 0 valores nulos
Video ID: 0 valores nulos
Published At: 0 valores nulos
Keyword: 0 valores nulos
Likes: 0 valores nulos
Comments: 0 valores nulos
Views: 0 valores nulos
Interaction: 0 valores nulos
Year: 0 valores nulos


In [20]:
# ------------------------------------------
# PASSO 16: Salvar df_video em formato parquet
# ------------------------------------------

if "_c0" in df_video.columns:
    df_video = df_video.drop("_c0")

try:
    df_video.write.mode("overwrite").parquet(
        "videos-tratados-parquet"
    )
    print("Arquivo videos-tratados-parquet salvo com sucesso!")

except Exception as e:
    print("Erro ao salvar df_video:")
    print(str(e))

Erro ao salvar df_video:
An error occurred while calling o369.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdi

In [ ]:
# ------------------------------------------
# PASSO 17: Salvar df_join_video_comments em parquet
# ------------------------------------------

if "_c0" in df_join_video_comments.columns:
    df_join_video_comments = df_join_video_comments.drop("_c0")

try:
    df_join_video_comments.write.mode("overwrite").parquet(
        "videos-comments-tratados-parquet"
    )
    print("Arquivo videos-comments-tratados-parquet salvo com sucesso!")

except Exception as e:
    print("Erro ao salvar df_join_video_comments:")
    print(str(e))

Py4JJavaError: An error occurred while calling o360.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

Logo a baixo é um teste feito para a linha 12 pois ela estava dando erros.

In [ ]:
print("COLUNAS DO DF_VIDEO:")
print(df_video.columns)

print("COLUNAS DO DF_COMENTARIO:")
print(df_comentario.columns)

In [ ]:
df_video.printSchema()
df_comentario.printSchema()

In [ ]:
try:
    teste_join = df_video.join(df_comentario, ["Video ID"], "inner")
    print("Junção realizada com sucesso!")
    print("Registros:", teste_join.count())

except Exception as e:
    print("ERRO ENCONTRADO:")
    print(str(e))
    